In [ ]:
# Fabric Notebook: NB_50_GOLD_POI_AI V8 - multi-language + maintainable category metadata
# Default Lakehouse: lh_vv_silver (MANDATORY)
# Gold targets use four-part workspace.lakehouse.schema.table names.
# READ : Silver + Gold reference tables
# WRITE: Gold ai_poi, ai_poi_rating, ai_poi_review, ai_enriched_poi

run_id = globals().get("run_id", "DEV_RUN")
STAGE = "_stage_poi_language"

spark.sql(f"CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS SELECT DISTINCT poi_id,poi_group_id,lang FROM {STAGE} WHERE run_id='{run_id}'")

try:
    spark.sql("ALTER TABLE sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi ADD COLUMNS (poi_group_id STRING)")
except Exception:
    pass
try:
    spark.sql("ALTER TABLE sgr_visitvn_dev.lh_vv_gold.dbo.ai_enriched_poi ADD COLUMNS (poi_group_id STRING, lang STRING)")
except Exception:
    pass

# -----------------------------------------------------------------------------
# SOURCE SCHEMA PREFLIGHT
# -----------------------------------------------------------------------------
# NB50 is intentionally pinned to lh_vv_silver, therefore Silver tables use
# single-part names. Validate the columns actually referenced below so schema
# drift fails with a concise error instead of a very large Spark plan.
_required_columns = {
    "poi_entity": [
        "poi_id","poi_group_id","lang","slug","poi_name","business_sector",
        "business_category","subcategory_tags","operating_status","is_active",
        "is_verified","canonical_source","partner_id","source_id","event_id",
        "_created_at","_updated_at","_is_current"
    ],
    "poi_address": [
        "poi_id","short_address","full_address","ward","district","city",
        "state_province","country_code","lat","lng"
    ],
    "poi_contact": [
        "poi_id","phone","email","website_url","booking_url","facebook_url",
        "instagram_url","tiktok_url"
    ],
    "poi_rating": [
        "poi_id","overall","total_reviews","score_label","breakdown_json",
        "sources_json","sentiment_avg","_updated_at"
    ],
    "poi_content": [
        "poi_id","locale","description_short","description_long","tags_text"
    ],
    "poi_media": [
        "poi_id","media_id","media_dedup_key","media_type","blob_url","thumbnail_url",
        "is_blessed","display_order","is_active","_updated_at"
    ],
    "poi_opening_hours": [
        "poi_id","timezone","is_24_7","open_now","periods_json"
    ],
    "poi_policy": ["poi_id","policy_json"],
    "poi_price": ["poi_id","currency","price_level","price_min","price_max"],
    "poi_amenity_current": [
        "poi_id","primary_type","resolved_amenities_json","ext_attributes_json"
    ],
    "poi_review": [
        "review_id","poi_id","source_name","source_review_id","language",
        "reviewer_name","rating_score","review_title","review_text",
        "review_media_urls","review_date","travel_type","helpful_count",
        "is_visible","response_text","responded_at","published_at",
        "_created_at","_updated_at"
    ],
    "poi_enrichment": [
        "enrichment_id","poi_id","poi_group_id","lang","model_name",
        "ref_business_sector","ref_type","description","time_to_spend_minutes",
        "best_time_type","best_time_from_month","best_time_to_month",
        "best_time_value","cost_vnd","cost_method","cost_confidence",
        "experiences_json","review_summary","reasoning_short","confidence",
        "needs_manual_review","story","usp_highlights_json",
        "local_tips_or_neighborhood","cancellation_policy","enriched_at",
        "is_current"
    ],
}

_schema_errors = []
for _table, _cols in _required_columns.items():
    try:
        _actual = set(spark.table(_table).columns)
        _missing = [c for c in _cols if c not in _actual]
        if _missing:
            _schema_errors.append(f"{_table}: missing columns {_missing}")
    except Exception as _e:
        _schema_errors.append(f"{_table}: cannot read table: {_e}")

if _schema_errors:
    raise RuntimeError(
        "NB50_SOURCE_SCHEMA_MISMATCH:\n" + "\n".join(_schema_errors)
    )

print("NB50 source schema preflight = OK")

# Gold target schema guard for ai_poi_review.
_expected_ai_review_cols = {
    "review_id","poi_id","source","source_review_id","language",
    "reviewer_name","rating","title","review_text","review_date",
    "review_trip_type","response_text","responded_at","media_urls",
    "helpful_votes","positive","negative","source_id","event_id",
    "created_at","updated_at"
}

_ai_review_target = "sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_review"
_ai_review_actual = set(spark.table(_ai_review_target).columns)
_ai_review_missing = sorted(_expected_ai_review_cols - _ai_review_actual)

if _ai_review_missing:
    raise RuntimeError(
        "NB50_TARGET_SCHEMA_MISMATCH ai_poi_review missing columns: "
        + str(_ai_review_missing)
    )

print("NB50 ai_poi_review target schema preflight = OK")


# -----------------------------------------------------------------------------
# MEDIA DEDUP CONTRACT (defensive downstream consumer for AI)
# -----------------------------------------------------------------------------
# NB20 is the ONLY authoritative writer of media business identity:
#   media_dedup_key = SHA256(source_name|source_id|media_id)
#
# NB50 does NOT recompute the key from photo_api_uri and does NOT use
# original_url as a business identity. It consumes the persisted NB20 key and
# defensively deduplicates per logical POI before selecting the primary image.
#
# Historical rows without a persisted key fall back to media_id only.
# -----------------------------------------------------------------------------
_media_cols = set(spark.table("poi_media").columns)
_has_media_dedup_key = "media_dedup_key" in _media_cols
_has_created_at = "_created_at" in _media_cols

if not _has_media_dedup_key:
    raise RuntimeError(
        "NB50_MEDIA_SCHEMA_MISMATCH: poi_media.media_dedup_key is required. "
        "Run the updated NB20 before NB50."
    )

_media_identity_expr = (
    "COALESCE(NULLIF(TRIM(m.media_dedup_key), ''), CONCAT('LEGACY_ID:', m.media_id))"
)
_media_identity_mode = "PERSISTED_MEDIA_DEDUP_KEY"
_media_created_order = "m._created_at ASC," if _has_created_at else ""

print(f"NB50 media dedup mode = {_media_identity_mode}")
print(
    "NB50 media contract = poi_id + persisted media_dedup_key "
    "(NB20 authoritative; media_id fallback only for historical rows)"
)

# ai_poi: preserve V1 feature projection, but use logical POI language and impacted gate.
# This source is deliberately Silver-centric so NB_40 and NB_50 can later run in parallel.
# NOTE: current poi_content schema has no local_tips column; ai_poi source emits NULL for
# local_tips here. Localized enrichment tips remain available in poi_enrichment.local_tips_or_neighborhood.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_ai_poi_src AS
WITH entity AS (
 SELECT e.* FROM poi_entity e JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
 WHERE e._is_current=true
), content_pick AS (
 SELECT * FROM (
  SELECT c.*,e.lang,ROW_NUMBER() OVER(PARTITION BY c.poi_id ORDER BY CASE WHEN c.locale=e.lang THEN 0 WHEN c.locale='vi' THEN 1 WHEN c.locale='en' THEN 2 ELSE 3 END) rn
  FROM poi_content c JOIN entity e ON e.poi_id=c.poi_id
 ) WHERE rn=1
), media_ranked AS (
 SELECT
   m.*,
   {_media_identity_expr} AS _media_identity,
   ROW_NUMBER() OVER (
     PARTITION BY m.poi_id, {_media_identity_expr}
     ORDER BY {_media_created_order} COALESCE(m.display_order,999999), m.media_id
   ) AS _dedup_rn
 FROM poi_media m
 JOIN v2_impacted_poi i ON i.poi_id=m.poi_id
 JOIN entity e ON e.poi_id=m.poi_id
 WHERE m.is_active=true AND m.media_type='photo'
), media_unique AS (
 SELECT * FROM media_ranked WHERE _dedup_rn=1
), media_pick AS (
 SELECT * FROM (
   SELECT m.*,
     ROW_NUMBER() OVER(
       PARTITION BY m.poi_id
       ORDER BY COALESCE(m.is_blessed,false) DESC,COALESCE(m.display_order,999999),m._updated_at DESC,m.media_id
     ) rn
   FROM media_unique m
 ) WHERE rn=1
)
SELECT
 e.poi_id,e.poi_group_id,e.lang,e.slug,e.poi_name,e.business_sector,e.business_category,to_json(e.subcategory_tags) subcategory_tags,
 e.operating_status,e.is_active,e.is_verified,e.canonical_source ingestion_source,e.partner_id,e.source_id,e.event_id,
 a.short_address,a.full_address,a.ward,a.district,a.city,a.state_province,a.country_code,a.lat,a.lng,oh.timezone AS timezone,
 c.phone,c.email,c.website_url,c.booking_url,c.facebook_url,c.instagram_url,c.tiktok_url,
 r.overall rating_overall,r.total_reviews rating_count,r.score_label,r.breakdown_json rating_breakdown,
 cp.description_short,cp.description_long,cp.tags_text,CAST(NULL AS STRING) AS local_tips,
 mp.blob_url primary_image_url,mp.thumbnail_url primary_thumbnail_url,
 oh.is_24_7,oh.open_now,oh.periods_json,pl.policy_json,
 pr.currency,pr.price_level,pr.price_min,pr.price_max,
 ac.primary_type,ac.resolved_amenities_json,ac.ext_attributes_json,
 e._created_at created_at,e._updated_at updated_at
FROM entity e
LEFT JOIN poi_address a ON a.poi_id=e.poi_id
LEFT JOIN poi_contact c ON c.poi_id=e.poi_id
LEFT JOIN poi_rating r ON r.poi_id=e.poi_id
LEFT JOIN content_pick cp ON cp.poi_id=e.poi_id
LEFT JOIN media_pick mp ON mp.poi_id=e.poi_id
LEFT JOIN poi_opening_hours oh ON oh.poi_id=e.poi_id
LEFT JOIN poi_policy pl ON pl.poi_id=e.poi_id
LEFT JOIN poi_price pr ON pr.poi_id=e.poi_id
LEFT JOIN poi_amenity_current ac ON ac.poi_id=e.poi_id
""")

# -----------------------------------------------------------------------------
# ai_poi - physical-schema-aware MERGE
#
# IMPORTANT:
#   v2_ai_poi_src is the canonical impacted multi-language source.
#   Persist it BEFORE sibling-map / ai_enriched_poi processing. Otherwise a
#   language present in Silver but absent from ai_poi can never reach
#   ai_enriched_poi.
#
# Compatibility:
#   - write only the intersection of source/target physical columns;
#   - do not alter legacy target-only columns;
#   - poi_id remains the business key because each language has its own poi_id.
# -----------------------------------------------------------------------------
_AI_POI_TBL = "sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi"

_ai_poi_target_cols = spark.table(_AI_POI_TBL).columns
_ai_poi_source_cols = spark.table("v2_ai_poi_src").columns
_ai_poi_source_col_set = set(_ai_poi_source_cols)

if "poi_id" not in _ai_poi_target_cols:
    raise RuntimeError("NB50_AI_POI_TARGET_SCHEMA_MISMATCH: missing poi_id")
if "poi_id" not in _ai_poi_source_col_set:
    raise RuntimeError("NB50_AI_POI_SOURCE_SCHEMA_MISMATCH: missing poi_id")

_ai_poi_common_cols = [
    c for c in _ai_poi_target_cols
    if c in _ai_poi_source_col_set
]
_ai_poi_update_cols = [
    c for c in _ai_poi_common_cols
    if c != "poi_id"
]

if not _ai_poi_update_cols:
    raise RuntimeError(
        "NB50_AI_POI_NO_UPDATE_COLUMNS: no writable non-key columns "
        "shared between v2_ai_poi_src and ai_poi"
    )

_ai_poi_update_sql = ",\n    ".join(
    [f"tgt.`{c}` = src.`{c}`" for c in _ai_poi_update_cols]
)
_ai_poi_insert_cols_sql = ", ".join(
    [f"`{c}`" for c in _ai_poi_common_cols]
)
_ai_poi_insert_vals_sql = ", ".join(
    [f"src.`{c}`" for c in _ai_poi_common_cols]
)

_ai_poi_merge_sql = f"""
MERGE INTO {_AI_POI_TBL} tgt
USING v2_ai_poi_src src
ON tgt.poi_id = src.poi_id

WHEN MATCHED THEN UPDATE SET
    {_ai_poi_update_sql}

WHEN NOT MATCHED THEN INSERT (
    {_ai_poi_insert_cols_sql}
)
VALUES (
    {_ai_poi_insert_vals_sql}
)
"""

spark.sql(_ai_poi_merge_sql)

# Hard invariant: every impacted language-specific POI materialized by the
# current source must now exist in ai_poi with the same poi_id/lang.
_ai_poi_missing = spark.sql(f"""
SELECT COUNT(*) AS c
FROM v2_ai_poi_src src
LEFT ANTI JOIN {_AI_POI_TBL} tgt
    ON tgt.poi_id = src.poi_id
   AND lower(trim(tgt.lang)) = lower(trim(src.lang))
""").first()["c"]

if _ai_poi_missing > 0:
    raise RuntimeError(
        "NB50_AI_POI_POST_MERGE_MISSING: "
        f"{_ai_poi_missing} source (poi_id, lang) rows are missing in ai_poi"
    )

print(
    "NB50 ai_poi multi-language MERGE = OK; "
    f"source_rows={spark.sql('SELECT COUNT(*) AS c FROM v2_ai_poi_src').first()['c']}"
)

# ai_poi_rating
spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_rating tgt
USING (
 SELECT md5(r.poi_id) rating_id,r.poi_id,r.overall,r.total_reviews total_rating,r.score_label,r.sources_json sources,r.breakdown_json breakdown,r.sentiment_avg,e.source_id,e.event_id,r._updated_at created_at,r._updated_at updated_at
 FROM poi_rating r JOIN v2_impacted_poi i ON i.poi_id=r.poi_id
 LEFT JOIN poi_entity e ON e.poi_id=r.poi_id AND e._is_current=true
) src ON tgt.poi_id=src.poi_id
WHEN MATCHED THEN UPDATE SET tgt.overall=src.overall,tgt.total_rating=src.total_rating,tgt.score_label=src.score_label,tgt.sources=src.sources,tgt.breakdown=src.breakdown,tgt.sentiment_avg=src.sentiment_avg,tgt.source_id=src.source_id,tgt.event_id=src.event_id,tgt.updated_at=src.updated_at
WHEN NOT MATCHED THEN INSERT(rating_id,poi_id,overall,total_rating,score_label,sources,breakdown,sentiment_avg,source_id,event_id,created_at,updated_at)
VALUES(src.rating_id,src.poi_id,src.overall,src.total_rating,src.score_label,src.sources,src.breakdown,src.sentiment_avg,src.source_id,src.event_id,src.created_at,src.updated_at)
""")

# -----------------------------------------------------------------------------
# ai_poi_review
#
# IMPORTANT:
# Physical Gold ai_poi_review keeps the legacy/V1 column names:
#   rating, title, review_trip_type, media_urls, helpful_votes, ...
#
# Silver poi_review uses normalized V2 names:
#   rating_score, review_title, travel_type, review_media_urls, helpful_count, ...
#
# Map explicitly here. Do NOT use rating_score/review_title/helpful_count as
# target column names because those columns do not exist in Gold ai_poi_review.
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_ai_poi_review_src AS
SELECT
    r.review_id,
    r.poi_id,
    r.source_name AS source,
    r.source_review_id,
    r.language,
    r.reviewer_name,
    r.rating_score AS rating,
    r.review_title AS title,
    r.review_text,
    r.review_date,
    r.travel_type AS review_trip_type,
    r.response_text,
    r.responded_at,
    to_json(r.review_media_urls) AS media_urls,
    r.helpful_count AS helpful_votes,
    CAST(NULL AS STRING) AS positive,
    CAST(NULL AS STRING) AS negative,
    e.source_id,
    e.event_id,
    COALESCE(
        r.published_at,
        CAST(r.review_date AS TIMESTAMP),
        r._created_at
    ) AS created_at,
    current_timestamp() AS updated_at
FROM poi_review r
INNER JOIN v2_impacted_poi i
    ON i.poi_id = r.poi_id
LEFT JOIN poi_entity e
    ON e.poi_id = r.poi_id
   AND e._is_current = true
WHERE r.is_visible = true
""")

spark.sql("""
MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_review tgt
USING v2_ai_poi_review_src src
ON tgt.review_id = src.review_id

WHEN MATCHED THEN UPDATE SET
    tgt.poi_id            = src.poi_id,
    tgt.source            = src.source,
    tgt.source_review_id  = src.source_review_id,
    tgt.language          = src.language,
    tgt.reviewer_name     = src.reviewer_name,
    tgt.rating            = src.rating,
    tgt.title             = src.title,
    tgt.review_text       = src.review_text,
    tgt.review_date       = src.review_date,
    tgt.review_trip_type  = src.review_trip_type,
    tgt.response_text     = src.response_text,
    tgt.responded_at      = src.responded_at,
    tgt.media_urls        = src.media_urls,
    tgt.helpful_votes     = src.helpful_votes,
    tgt.positive          = src.positive,
    tgt.negative          = src.negative,
    tgt.source_id         = src.source_id,
    tgt.event_id          = src.event_id,
    tgt.updated_at        = src.updated_at

WHEN NOT MATCHED THEN INSERT (
    review_id,
    poi_id,
    source,
    source_review_id,
    language,
    reviewer_name,
    rating,
    title,
    review_text,
    review_date,
    review_trip_type,
    response_text,
    responded_at,
    media_urls,
    helpful_votes,
    positive,
    negative,
    source_id,
    event_id,
    created_at,
    updated_at
)
VALUES (
    src.review_id,
    src.poi_id,
    src.source,
    src.source_review_id,
    src.language,
    src.reviewer_name,
    src.rating,
    src.title,
    src.review_text,
    src.review_date,
    src.review_trip_type,
    src.response_text,
    src.responded_at,
    src.media_urls,
    src.helpful_votes,
    src.positive,
    src.negative,
    src.source_id,
    src.event_id,
    src.created_at,
    src.updated_at
)
""")

# -----------------------------------------------------------------------------
# ai_enriched_poi
# -----------------------------------------------------------------------------
# Business grain:
#   1 row = 1 (poi_id, lang)
#
# Sibling language mapping:
#   poi_vi / poi_en / poi_ko are POI IDs of the same poi_group_id.
#   Mapping is built from Gold ai_poi so every sibling reference points to an
#   AI-serving POI that is actually materialized.
#
# Enrichment:
#   source of truth = Silver poi_enrichment
#   choose latest row per poi_id among is_current = true
#
# Target:
#   Gold ai_enriched_poi
#   MERGE key = poi_id + lang
#
# IMPORTANT:
#   - sibling map uses the whole ai_poi table, not only impacted rows;
#   - duplicate (poi_group_id, lang) is treated as a hard invariant violation;
#   - target MERGE is physical-schema aware: only columns actually present in
#     ai_enriched_poi are written, so legacy target columns remain compatible.
# -----------------------------------------------------------------------------

AI_POI_TBL = "sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi"
AI_ENRICHED_TBL = "sgr_visitvn_dev.lh_vv_gold.dbo.ai_enriched_poi"

# -----------------------------------------------------------------------------
# 5.1 Invariant: one poi_group_id + one normalized language -> max one poi_id
# -----------------------------------------------------------------------------
_ai_sibling_conflicts = spark.sql(f"""
SELECT
    poi_group_id,
    lower(trim(lang)) AS lang,
    COUNT(DISTINCT poi_id) AS poi_count
FROM {AI_POI_TBL}
WHERE poi_group_id IS NOT NULL
  AND lower(trim(lang)) IN ('vi','en','ko')
GROUP BY poi_group_id, lower(trim(lang))
HAVING COUNT(DISTINCT poi_id) > 1
""")

_ai_sibling_conflict_count = _ai_sibling_conflicts.count()

if _ai_sibling_conflict_count > 0:
    print("NB50 AI sibling conflict sample:")
    _ai_sibling_conflicts.show(100, truncate=False)
    raise RuntimeError(
        "NB50_AI_POI_SIBLING_CONFLICT: "
        f"{_ai_sibling_conflict_count} (poi_group_id, lang) groups have >1 poi_id"
    )


# -----------------------------------------------------------------------------
# 5.2 Build sibling language map from ALL materialized ai_poi rows
# -----------------------------------------------------------------------------
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_ai_poi_sibling_map AS
SELECT
    poi_group_id,
    MAX(CASE WHEN lower(trim(lang)) = 'vi' THEN poi_id END) AS poi_vi,
    MAX(CASE WHEN lower(trim(lang)) = 'en' THEN poi_id END) AS poi_en,
    MAX(CASE WHEN lower(trim(lang)) = 'ko' THEN poi_id END) AS poi_ko
FROM {AI_POI_TBL}
WHERE poi_group_id IS NOT NULL
GROUP BY poi_group_id
""")

# -----------------------------------------------------------------------------
# 5.3 Latest current enrichment per poi_id
# -----------------------------------------------------------------------------
spark.sql("""
CREATE OR REPLACE TEMP VIEW v2_ai_latest_enrichment AS
SELECT *
FROM (
    SELECT
        e.*,
        ROW_NUMBER() OVER (
            PARTITION BY e.poi_id
            ORDER BY
                e.enriched_at DESC NULLS LAST,
                e._updated_at DESC NULLS LAST,
                e.enrichment_id DESC
        ) AS rn
    FROM poi_enrichment e
    INNER JOIN v2_impacted_poi i
        ON i.poi_id = e.poi_id
    WHERE e.is_current = true
) x
WHERE rn = 1
""")

# -----------------------------------------------------------------------------
# 5.4 Build ai_enriched source
# -----------------------------------------------------------------------------
# Start from impacted AI POIs. This guarantees one serving row per logical
# language-specific POI while still allowing sibling IDs from other languages.
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW v2_ai_enriched_src AS
SELECT
    a.*,

    -- poi_types is inherited from ai_poi and is always materialized as 'PLACE'.

    -- Sibling logical POI IDs in the same poi_group_id.
    s.poi_vi,
    s.poi_en,
    s.poi_ko,

    -- ai_enriched_poi legacy/serving schema mapping.
    -- These aliases intentionally match the physical target column names so
    -- the schema-aware MERGE below will actually persist them.
    e.enrichment_id                 AS enrichment_id,
    CAST(NULL AS STRING)            AS enrichment_job_id,
    CAST(NULL AS STRING)            AS enrichment_status,
    CAST(NULL AS STRING)            AS validation_status,
    e.model_name                    AS model_name,
    e._updated_at                   AS source_updated_at,
    e.enriched_at                   AS enriched_at,

    e.description                   AS ref_description,
    e.time_to_spend_minutes         AS ref_time_to_spend_minutes,
    e.best_time_type                AS ref_best_time_type,
    e.best_time_from_month          AS ref_best_time_from_month,
    e.best_time_to_month            AS ref_best_time_to_month,
    e.cost_vnd                      AS ref_cost_vnd,
    e.cost_method                   AS ref_cost_method,
    e.cost_confidence               AS ref_cost_confidence,
    e.experiences_json              AS ref_experiences,
    e.review_summary                AS ref_review_summary,
    e.reasoning_short               AS ref_reasoning_short,

    -- Keep additional V2 enrichment fields when the physical target supports them.
    e.poi_group_id                  AS enrichment_poi_group_id,
    e.lang                          AS enrichment_lang,
    e.ref_business_sector           AS enrichment_ref_business_sector,
    e.ref_type                      AS enrichment_ref_type,
    e.best_time_value               AS enrichment_best_time_value,
    e.confidence                    AS enrichment_confidence,
    e.needs_manual_review           AS enrichment_needs_manual_review,
    e.story                         AS enrichment_story,
    e.usp_highlights_json           AS enrichment_usp_highlights_json,
    e.local_tips_or_neighborhood    AS enrichment_local_tips_or_neighborhood,
    e.cancellation_policy           AS enrichment_cancellation_policy

FROM {AI_POI_TBL} a
INNER JOIN v2_impacted_poi i
    ON i.poi_id = a.poi_id
LEFT JOIN v2_ai_poi_sibling_map s
    ON s.poi_group_id = a.poi_group_id
LEFT JOIN v2_ai_latest_enrichment e
    ON e.poi_id = a.poi_id
""")

# -----------------------------------------------------------------------------
# 5.5 BUSINESS_CATEGORY REMAP METADATA - ONLY for ai_enriched_poi
# -----------------------------------------------------------------------------
# MAINTENANCE CONTRACT:
#   - This dictionary is the ONLY metadata block to edit when adding/removing
#     business_category mappings.
#   - Key   = source ai_poi.business_category (exact match after trim).
#   - Value = serving ai_enriched_poi.business_category.
#   - Unmapped source values are preserved unchanged.
#   - ai_poi.business_category is NOT modified by this mapping.
#   - Duplicate keys are impossible in a Python dict literal; keep each source
#     category declared once.
#
# Source of truth for this version: user-supplied business-category mapping MD.
# Mapping entries: 132
# -----------------------------------------------------------------------------
AI_ENRICHED_POI_BUSINESS_CATEGORY_REMAP = {
    'Hotel': 'hotel',
    'Extended Stay Hotel': 'extended_stay_hotel',
    'Inn': 'inn',
    'Resort': 'resort_hotel',
    'Hostel': 'hostel',
    'Motel': 'motel',
    'Homestay': 'guest_house',
    'Farmstay': 'farmstay',
    'Campground/Glamping': 'campground',
    'Camping Cabin': 'camping_cabin',
    'Bed & Breakfast': 'bed_and_breakfast',
    'Cottage': 'cottage',
    'Mobile Home Park': 'mobile_home_park',
    'Accommodation': 'lodging',
    'Vietnamese': 'vietnamese_restaurant',
    'Chinese': 'chinese_restaurant',
    'Japanese': 'japanese_restaurant',
    'Korean': 'korean_restaurant',
    'Thai': 'thai_restaurant',
    'French': 'french_restaurant',
    'Italian': 'italian_restaurant',
    'American': 'american_restaurant',
    'Western': 'western_restaurant',
    'International': 'internation_restaurant',
    'Fusion': 'fusion_restaurant',
    'Bakery & Dessert': 'bakery_dessert',
    'Bar & Pub/Nightlife': 'night_club',
    'Fast Food': 'fast_food',
    'Vegan Restaurant': 'vegetarian_restaurant',
    'Seafood Restaurant': 'seafood_restaurant',
    'Café': 'cafe',
    'Hot Pot': 'hot_pot_restaurant',
    'Barbecue & Grill': 'barbecue_restaurant',
    'Noodle': 'noodle_shop',
    'Fine Dining': 'fine_dining_restaurant',
    'Buffet': 'buffet_restaurant',
    'Brunch & Breakfast': 'brunch_restaurant',
    'Halal': 'halal_restaurant',
    'Gastropub & Brewery': 'gastropub',
    'F&B': 'f_n_b',
    'Day Spa': 'spa',
    'Therapeutic Massage': 'therapeutic_massage',
    'Clinic-Aesthetic Spa': 'clinic_aesthetic_spa',
    'Blind Massage': 'blind_massage',
    'Onsen-Jjimjilbang': 'onsen',
    'Hair Salon': 'hair_salon',
    'Nail-Lash Salon': 'nail_salon',
    'Gym-Fitness': 'fitness_center',
    'Yoga-Pilates Studio': 'yoga_pilates_studio',
    'Meditation Center': 'meditation_center',
    'Beauty Salon': 'beauty_salon',
    'Hair Removal Service': 'hair_removal_service',
    'Tanning Studio': 'tanning_studio',
    'Barber Shop': 'barber_shop',
    'Tattoo & Piercing': 'body_piercing_shop',
    'Swimming Pool': 'swimming_pool',
    'Local Specialty': 'local_specialty',
    'Souvenirs & Handicrafts': 'souvenir_handicraft',
    'Fashion & Accessories': 'fashion_accessories',
    'Jewelry & Gems': 'jewelry_store',
    'Cosmetics & Beauty': 'cosmetics_store',
    'Books & Stationery': 'book_store',
    'Mall - Supermarket': 'shopping_mall',
    'Convenience Store': 'convenience_store',
    'Local/Night Market': 'local_market',
    'Sporting Goods': 'sporting_goods_store',
    'Toy Store': 'toy_store',
    'Luggage Store': 'luggage_store',
    'Florist': 'florist',
    'Home & Decor': 'home_n_decor',
    'Shopping & Retail': 'store',
    'Theme Park': 'theme_park',
    'Water Park': 'water_park',
    'Museum': 'museum',
    'Historical Site': 'historical_place',
    'Zoo-Aquarium': 'zoo_and_aquarium',
    'Nature Reserve': 'national_park',
    'Architecture-Landmark': 'landmark',
    'Art Show-Performance': 'art_show_performance',
    'Traditional Village': 'traditional_village',
    'Religious Site': 'religious_site',
    'Casino': 'casino',
    'Art Gallery': 'art_gallery',
    'Cultural Center': 'cultural_center',
    'Indoor Entertainment': 'indoor_entertainment',
    'Beach': 'beach',
    'Park': 'park',
    'National Park': 'national_park',
    'Hiking Area': 'hiking_area',
    'Viewpoint': 'viewpoint',
    'Observation Deck': 'observation_deck',
    'Waterfall': 'waterfall',
    'Cave': 'cave',
    'Botanical Garden': 'botanical_garden',
    'Stadium': 'stadium',
    'Motorbike Rental': 'motocycle_rental_agency',
    'Self-drive Car Rental': 'self_drive_car_rental',
    'Car with Driver': 'chauffeur_car_rental',
    'Bicycle-EV Rental': 'bicycle_ev_rental',
    'Campervan Rental': 'campervan_rental',
    'Boat Rental': 'boat_rental',
    'Music & Entertainment': 'music_entertainment_event',
    'Sports': 'sports_event',
    'Education & Business': 'education_business_event',
    'Festival & Community': 'festival_community_event',
    'Arts & Culture': 'arts_culture_event',
    'Sleeper Bus': 'sleeper_bus_service',
    'Limousine Van': 'limousine_service',
    'Speedboat': 'speedboat_service',
    'Ferry': 'ferry_service',
    'Train': 'train_service',
    'Flight': 'flight',
    'Traditional Taxi': 'taxi_service',
    'Chauffeur & Charter Company': 'chauffeur_charter_service',
    'Tech-hailing Platform': 'ride_hailing_platform',
    'Airport Limousine Service': 'airport_limousine_service',
    'Italian Restaurant': 'italian_restaurant',
    'Japanese Restaurant': 'japanese_restaurant',
    'American Restaurant': 'american_restaurant',
    'French Restaurant': 'french_restaurant',
    'Thai Restaurant': 'thai_restaurant',
    'Korean Restaurant': 'korean_restaurant',
    'Chinese Restaurant': 'chinese_restaurant',
    'International Restaurant': 'internation_restaurant',
    'Western Restaurant': 'western_restaurant',
    'Vietnamese Restaurant': 'vietnamese_restaurant',
    'Local / Night Market': 'local_market',
    'Bar & Pub / Nightlife': 'night_club',
    'Mall-Supermarket': 'shopping_mall',
    'Spa': 'spa',
    'Barbershop': 'barber_shop',
    'Other': 'other',
}

# Metadata validation: fail early on malformed maintenance changes.
_invalid_business_category_mapping = [
    (k, v)
    for k, v in AI_ENRICHED_POI_BUSINESS_CATEGORY_REMAP.items()
    if not isinstance(k, str)
    or not isinstance(v, str)
    or not k.strip()
    or not v.strip()
]

if _invalid_business_category_mapping:
    raise RuntimeError(
        "NB50_AI_ENRICHED_BUSINESS_CATEGORY_MAPPING_INVALID: "
        + str(_invalid_business_category_mapping[:20])
    )

print(
    "NB50 business_category metadata loaded; "
    f"mapping_entries={len(AI_ENRICHED_POI_BUSINESS_CATEGORY_REMAP)}"
)

# -----------------------------------------------------------------------------
# 5.5.1 Apply metadata mapping
# -----------------------------------------------------------------------------
from pyspark.sql import functions as F

_mapping_expr_items = []
for _source_category, _target_category in AI_ENRICHED_POI_BUSINESS_CATEGORY_REMAP.items():
    _mapping_expr_items.extend([
        F.lit(_source_category),
        F.lit(_target_category),
    ])

_AI_ENRICHED_BUSINESS_CATEGORY_REMAP_EXPR = F.create_map(*_mapping_expr_items)

_ai_enriched_df = spark.table("v2_ai_enriched_src")
if "business_category" not in _ai_enriched_df.columns:
    raise RuntimeError(
        "NB50_AI_ENRICHED_BUSINESS_CATEGORY_MISSING: "
        "v2_ai_enriched_src has no business_category"
    )

# Materialize before replacing the original temp-view name.
# This prevents the lazy-plan cycle:
# v2_ai_enriched_src -> mapped DF -> v2_ai_enriched_src.
_ai_enriched_mapped_df = _ai_enriched_df.withColumn(
    "business_category",
    F.coalesce(
        F.element_at(
            _AI_ENRICHED_BUSINESS_CATEGORY_REMAP_EXPR,
            F.trim(F.col("business_category"))
        ),
        F.col("business_category")
    )
).cache()

_ai_enriched_mapped_count = _ai_enriched_mapped_df.count()

spark.catalog.dropTempView("v2_ai_enriched_src")
_ai_enriched_mapped_df.createOrReplaceTempView("v2_ai_enriched_src")

print(
    "NB50 ai_enriched_poi business_category remap enabled; "
    f"mapped_source_rows={_ai_enriched_mapped_count}; "
    f"mapping_entries={len(AI_ENRICHED_POI_BUSINESS_CATEGORY_REMAP)}; "
    "unmapped=preserve_original"
)

# -----------------------------------------------------------------------------
# 5.6 Source quality checks
# -----------------------------------------------------------------------------
_ai_enriched_cols = spark.table("v2_ai_enriched_src").columns

_ai_enriched_dups = sorted({
    c for c in _ai_enriched_cols
    if _ai_enriched_cols.count(c) > 1
})

if _ai_enriched_dups:
    raise RuntimeError(
        "NB50_AI_ENRICHED_DUPLICATE_COLUMNS: " + str(_ai_enriched_dups)
    )

_ai_enriched_key_dups = spark.sql("""
SELECT COUNT(*) AS c
FROM (
    SELECT poi_id, lang
    FROM v2_ai_enriched_src
    GROUP BY poi_id, lang
    HAVING COUNT(*) > 1
) x
""").first()["c"]

if _ai_enriched_key_dups > 0:
    raise RuntimeError(
        "NB50_AI_ENRICHED_DUPLICATE_KEY: "
        f"{_ai_enriched_key_dups} duplicate (poi_id, lang) keys"
    )

_ai_enriched_source_count = spark.sql(
    "SELECT COUNT(*) AS c FROM v2_ai_enriched_src"
).first()["c"]

# -----------------------------------------------------------------------------
# 5.6 Physical-schema-aware MERGE into ai_enriched_poi
# -----------------------------------------------------------------------------
# ai_enriched_poi is a legacy/serving table whose physical schema may contain
# only a subset of v2_ai_enriched_src. Build the MERGE from the actual target
# schema instead of hard-coding logical column names.
#
# This also makes poi_vi / poi_en / poi_ko populate automatically whenever
# those columns exist in the physical target.
_target_cols = spark.table(AI_ENRICHED_TBL).columns
_source_cols = set(_ai_enriched_cols)

_required_target_keys = {"poi_id", "lang"}
_missing_target_keys = sorted(_required_target_keys - set(_target_cols))
if _missing_target_keys:
    raise RuntimeError(
        "NB50_AI_ENRICHED_TARGET_SCHEMA_MISMATCH: "
        f"missing merge key columns {_missing_target_keys}"
    )

_missing_source_keys = sorted(_required_target_keys - _source_cols)
if _missing_source_keys:
    raise RuntimeError(
        "NB50_AI_ENRICHED_SOURCE_SCHEMA_MISMATCH: "
        f"missing merge key columns {_missing_source_keys}"
    )

# Writable columns are the intersection of source and physical target.
_common_cols = [c for c in _target_cols if c in _source_cols]

# Never rewrite merge keys in UPDATE.
_update_cols = [c for c in _common_cols if c not in ("poi_id", "lang")]

# INSERT requires at least the merge keys.
_insert_cols = _common_cols

if not _update_cols:
    raise RuntimeError(
        "NB50_AI_ENRICHED_NO_UPDATE_COLUMNS: no writable non-key columns "
        "shared between source and target"
    )

# Helpful diagnostics, especially for schema drift.
_target_only = [c for c in _target_cols if c not in _source_cols]
_source_only = [c for c in _ai_enriched_cols if c not in set(_target_cols)]


_update_set_sql = ",\n    ".join(
    [f"tgt.`{c}` = src.`{c}`" for c in _update_cols]
)

_insert_cols_sql = ", ".join([f"`{c}`" for c in _insert_cols])
_insert_vals_sql = ", ".join([f"src.`{c}`" for c in _insert_cols])

_ai_enriched_merge_sql = f"""
MERGE INTO {AI_ENRICHED_TBL} tgt
USING v2_ai_enriched_src src
ON tgt.poi_id = src.poi_id
AND tgt.lang = src.lang

WHEN MATCHED THEN UPDATE SET
    {_update_set_sql}

WHEN NOT MATCHED THEN INSERT (
    {_insert_cols_sql}
)
VALUES (
    {_insert_vals_sql}
)
"""

spark.sql(_ai_enriched_merge_sql)

# -----------------------------------------------------------------------------
# 5.7 Post-merge invariants
# -----------------------------------------------------------------------------
_missing_ai_enriched = spark.sql(f"""
SELECT COUNT(*) AS c
FROM v2_ai_enriched_src src
LEFT ANTI JOIN {AI_ENRICHED_TBL} tgt
    ON tgt.poi_id = src.poi_id
   AND tgt.lang = src.lang
""").first()["c"]

if _missing_ai_enriched > 0:
    raise RuntimeError(
        "NB50_AI_ENRICHED_POST_MERGE_MISSING: "
        f"{_missing_ai_enriched} source rows are missing in target"
    )

# Hard invariant independent of v2_ai_enriched_src:
# every impacted language-specific POI must be materialized in ai_enriched_poi.
_missing_impacted_ai_enriched = spark.sql(f"""
SELECT COUNT(*) AS c
FROM v2_impacted_poi i
LEFT ANTI JOIN {AI_ENRICHED_TBL} tgt
    ON tgt.poi_id = i.poi_id
   AND lower(trim(tgt.lang)) = lower(trim(i.lang))
""").first()["c"]

if _missing_impacted_ai_enriched > 0:
    raise RuntimeError(
        "NB50_AI_ENRICHED_IMPACTED_LANGUAGE_MISSING: "
        f"{_missing_impacted_ai_enriched} impacted (poi_id, lang) rows "
        "are missing in ai_enriched_poi"
    )

# Verify sibling values were persisted when the physical target supports them.
_sibling_target_cols = {"poi_vi", "poi_en", "poi_ko"} & set(_target_cols)

if sibling_cols := sorted(_sibling_target_cols):
    _sibling_compare = " OR ".join(
        [
            f"NOT (tgt.`{c}` <=> src.`{c}`)"
            for c in sibling_cols
        ]
    )

    _sibling_mismatch = spark.sql(f"""
    SELECT COUNT(*) AS c
    FROM v2_ai_enriched_src src
    INNER JOIN {AI_ENRICHED_TBL} tgt
        ON tgt.poi_id = src.poi_id
       AND tgt.lang = src.lang
    WHERE {_sibling_compare}
    """).first()["c"]

    if _sibling_mismatch > 0:
        raise RuntimeError(
            "NB50_AI_ENRICHED_SIBLING_MISMATCH: "
            f"{_sibling_mismatch} target rows have incorrect sibling IDs"
        )



notebookutils.notebook.exit("SUCCESS")


In [ ]:
# # Fabric Notebook: NB_50_GOLD_POI_AI
# # Default Lakehouse: lh_vv_silver (MANDATORY)
# # Gold targets use four-part workspace.lakehouse.schema.table names.
# # READ : Silver + Gold reference tables
# # WRITE: Gold ai_poi, ai_poi_rating, ai_poi_review, ai_enriched_poi

# run_id = globals().get("run_id", "DEV_RUN")
# STAGE = "_stage_poi_language"

# spark.sql(f"CREATE OR REPLACE TEMP VIEW v2_impacted_poi AS SELECT DISTINCT poi_id,poi_group_id,lang FROM {STAGE} WHERE run_id='{run_id}'")

# try:
#     spark.sql("ALTER TABLE sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi ADD COLUMNS (poi_group_id STRING)")
# except Exception:
#     pass
# try:
#     spark.sql("ALTER TABLE sgr_visitvn_dev.lh_vv_gold.dbo.ai_enriched_poi ADD COLUMNS (poi_group_id STRING, lang STRING)")
# except Exception:
#     pass

# # -----------------------------------------------------------------------------
# # SOURCE SCHEMA PREFLIGHT
# # -----------------------------------------------------------------------------
# # NB50 is intentionally pinned to lh_vv_silver, therefore Silver tables use
# # single-part names. Validate the columns actually referenced below so schema
# # drift fails with a concise error instead of a very large Spark plan.
# _required_columns = {
#     "poi_entity": [
#         "poi_id","poi_group_id","lang","slug","poi_name","business_sector",
#         "business_category","subcategory_tags","operating_status","is_active",
#         "is_verified","canonical_source","partner_id","source_id","event_id",
#         "_created_at","_updated_at","_is_current"
#     ],
#     "poi_address": [
#         "poi_id","short_address","full_address","ward","district","city",
#         "state_province","country_code","lat","lng"
#     ],
#     "poi_contact": [
#         "poi_id","phone","email","website_url","booking_url","facebook_url",
#         "instagram_url","tiktok_url"
#     ],
#     "poi_rating": [
#         "poi_id","overall","total_reviews","score_label","breakdown_json",
#         "sources_json","sentiment_avg","_updated_at"
#     ],
#     "poi_content": [
#         "poi_id","locale","description_short","description_long","tags_text"
#     ],
#     "poi_media": [
#         "poi_id","media_id","media_dedup_key","media_type","blob_url","thumbnail_url",
#         "is_blessed","display_order","is_active","_updated_at"
#     ],
#     "poi_opening_hours": [
#         "poi_id","timezone","is_24_7","open_now","periods_json"
#     ],
#     "poi_policy": ["poi_id","policy_json"],
#     "poi_price": ["poi_id","currency","price_level","price_min","price_max"],
#     "poi_amenity_current": [
#         "poi_id","primary_type","resolved_amenities_json","ext_attributes_json"
#     ],
#     "poi_review": [
#         "review_id","poi_id","source_name","source_review_id","language",
#         "reviewer_name","rating_score","review_title","review_text",
#         "review_media_urls","review_date","travel_type","helpful_count",
#         "is_visible","response_text","responded_at","published_at",
#         "_created_at","_updated_at"
#     ],
#     "poi_enrichment": [
#         "enrichment_id","poi_id","poi_group_id","lang","model_name",
#         "ref_business_sector","ref_type","description","time_to_spend_minutes",
#         "best_time_type","best_time_from_month","best_time_to_month",
#         "best_time_value","cost_vnd","cost_method","cost_confidence",
#         "experiences_json","review_summary","reasoning_short","confidence",
#         "needs_manual_review","story","usp_highlights_json",
#         "local_tips_or_neighborhood","cancellation_policy","enriched_at",
#         "is_current"
#     ],
# }

# _schema_errors = []
# for _table, _cols in _required_columns.items():
#     try:
#         _actual = set(spark.table(_table).columns)
#         _missing = [c for c in _cols if c not in _actual]
#         if _missing:
#             _schema_errors.append(f"{_table}: missing columns {_missing}")
#     except Exception as _e:
#         _schema_errors.append(f"{_table}: cannot read table: {_e}")

# if _schema_errors:
#     raise RuntimeError(
#         "NB50_SOURCE_SCHEMA_MISMATCH:\n" + "\n".join(_schema_errors)
#     )

# print("NB50 source schema preflight = OK")

# # Gold target schema guard for ai_poi_review.
# _expected_ai_review_cols = {
#     "review_id","poi_id","source","source_review_id","language",
#     "reviewer_name","rating","title","review_text","review_date",
#     "review_trip_type","response_text","responded_at","media_urls",
#     "helpful_votes","positive","negative","source_id","event_id",
#     "created_at","updated_at"
# }

# _ai_review_target = "sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_review"
# _ai_review_actual = set(spark.table(_ai_review_target).columns)
# _ai_review_missing = sorted(_expected_ai_review_cols - _ai_review_actual)

# if _ai_review_missing:
#     raise RuntimeError(
#         "NB50_TARGET_SCHEMA_MISMATCH ai_poi_review missing columns: "
#         + str(_ai_review_missing)
#     )

# print("NB50 ai_poi_review target schema preflight = OK")


# # -----------------------------------------------------------------------------
# # MEDIA DEDUP CONTRACT (defensive downstream consumer for AI)
# # -----------------------------------------------------------------------------
# # NB20 is the ONLY authoritative writer of media business identity:
# #   media_dedup_key = SHA256(source_name|source_id|media_id)
# #
# # NB50 does NOT recompute the key from photo_api_uri and does NOT use
# # original_url as a business identity. It consumes the persisted NB20 key and
# # defensively deduplicates per logical POI before selecting the primary image.
# #
# # Historical rows without a persisted key fall back to media_id only.
# # -----------------------------------------------------------------------------
# _media_cols = set(spark.table("poi_media").columns)
# _has_media_dedup_key = "media_dedup_key" in _media_cols
# _has_created_at = "_created_at" in _media_cols

# if not _has_media_dedup_key:
#     raise RuntimeError(
#         "NB50_MEDIA_SCHEMA_MISMATCH: poi_media.media_dedup_key is required. "
#         "Run the updated NB20 before NB50."
#     )

# _media_identity_expr = (
#     "COALESCE(NULLIF(TRIM(m.media_dedup_key), ''), CONCAT('LEGACY_ID:', m.media_id))"
# )
# _media_identity_mode = "PERSISTED_MEDIA_DEDUP_KEY"
# _media_created_order = "m._created_at ASC," if _has_created_at else ""

# print(f"NB50 media dedup mode = {_media_identity_mode}")
# print(
#     "NB50 media contract = poi_id + persisted media_dedup_key "
#     "(NB20 authoritative; media_id fallback only for historical rows)"
# )

# # ai_poi: preserve V1 feature projection, but use logical POI language and impacted gate.
# # This source is deliberately Silver-centric so NB_40 and NB_50 can later run in parallel.
# # NOTE: current poi_content schema has no local_tips column; ai_poi source emits NULL for
# # local_tips here. Localized enrichment tips remain available in poi_enrichment.local_tips_or_neighborhood.
# spark.sql(f"""
# CREATE OR REPLACE TEMP VIEW v2_ai_poi_src AS
# WITH entity AS (
#  SELECT e.* FROM poi_entity e JOIN v2_impacted_poi i ON i.poi_id=e.poi_id
#  WHERE e._is_current=true
# ), content_pick AS (
#  SELECT * FROM (
#   SELECT c.*,e.lang,ROW_NUMBER() OVER(PARTITION BY c.poi_id ORDER BY CASE WHEN c.locale=e.lang THEN 0 WHEN c.locale='vi' THEN 1 WHEN c.locale='en' THEN 2 ELSE 3 END) rn
#   FROM poi_content c JOIN entity e ON e.poi_id=c.poi_id
#  ) WHERE rn=1
# ), media_ranked AS (
#  SELECT
#    m.*,
#    {_media_identity_expr} AS _media_identity,
#    ROW_NUMBER() OVER (
#      PARTITION BY m.poi_id, {_media_identity_expr}
#      ORDER BY {_media_created_order} COALESCE(m.display_order,999999), m.media_id
#    ) AS _dedup_rn
#  FROM poi_media m
#  JOIN v2_impacted_poi i ON i.poi_id=m.poi_id
#  JOIN entity e ON e.poi_id=m.poi_id
#  WHERE m.is_active=true AND m.media_type='photo'
# ), media_unique AS (
#  SELECT * FROM media_ranked WHERE _dedup_rn=1
# ), media_pick AS (
#  SELECT * FROM (
#    SELECT m.*,
#      ROW_NUMBER() OVER(
#        PARTITION BY m.poi_id
#        ORDER BY COALESCE(m.is_blessed,false) DESC,COALESCE(m.display_order,999999),m._updated_at DESC,m.media_id
#      ) rn
#    FROM media_unique m
#  ) WHERE rn=1
# )
# SELECT
#  e.poi_id,e.poi_group_id,e.lang,e.slug,e.poi_name,e.business_sector,e.business_category,to_json(e.subcategory_tags) subcategory_tags,
#  e.operating_status,e.is_active,e.is_verified,e.canonical_source ingestion_source,e.partner_id,e.source_id,e.event_id,
#  a.short_address,a.full_address,a.ward,a.district,a.city,a.state_province,a.country_code,a.lat,a.lng,oh.timezone AS timezone,
#  c.phone,c.email,c.website_url,c.booking_url,c.facebook_url,c.instagram_url,c.tiktok_url,
#  r.overall rating_overall,r.total_reviews rating_count,r.score_label,r.breakdown_json rating_breakdown,
#  cp.description_short,cp.description_long,cp.tags_text,CAST(NULL AS STRING) AS local_tips,
#  mp.blob_url primary_image_url,mp.thumbnail_url primary_thumbnail_url,
#  oh.is_24_7,oh.open_now,oh.periods_json,pl.policy_json,
#  pr.currency,pr.price_level,pr.price_min,pr.price_max,
#  ac.primary_type,ac.resolved_amenities_json,ac.ext_attributes_json,
#  e._created_at created_at,e._updated_at updated_at
# FROM entity e
# LEFT JOIN poi_address a ON a.poi_id=e.poi_id
# LEFT JOIN poi_contact c ON c.poi_id=e.poi_id
# LEFT JOIN poi_rating r ON r.poi_id=e.poi_id
# LEFT JOIN content_pick cp ON cp.poi_id=e.poi_id
# LEFT JOIN media_pick mp ON mp.poi_id=e.poi_id
# LEFT JOIN poi_opening_hours oh ON oh.poi_id=e.poi_id
# LEFT JOIN poi_policy pl ON pl.poi_id=e.poi_id
# LEFT JOIN poi_price pr ON pr.poi_id=e.poi_id
# LEFT JOIN poi_amenity_current ac ON ac.poi_id=e.poi_id
# """)

# # IMPORTANT: adapt this MERGE column list to the physical ai_poi schema if V1 differs.
# # Existing V1 already MERGEs ai_poi by poi_id; keep the same physical columns and add poi_group_id/lang.

# # ai_poi_rating
# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_rating tgt
# USING (
#  SELECT md5(r.poi_id) rating_id,r.poi_id,r.overall,r.total_reviews total_rating,r.score_label,r.sources_json sources,r.breakdown_json breakdown,r.sentiment_avg,e.source_id,e.event_id,r._updated_at created_at,r._updated_at updated_at
#  FROM poi_rating r JOIN v2_impacted_poi i ON i.poi_id=r.poi_id
#  LEFT JOIN poi_entity e ON e.poi_id=r.poi_id AND e._is_current=true
# ) src ON tgt.poi_id=src.poi_id
# WHEN MATCHED THEN UPDATE SET tgt.overall=src.overall,tgt.total_rating=src.total_rating,tgt.score_label=src.score_label,tgt.sources=src.sources,tgt.breakdown=src.breakdown,tgt.sentiment_avg=src.sentiment_avg,tgt.source_id=src.source_id,tgt.event_id=src.event_id,tgt.updated_at=src.updated_at
# WHEN NOT MATCHED THEN INSERT(rating_id,poi_id,overall,total_rating,score_label,sources,breakdown,sentiment_avg,source_id,event_id,created_at,updated_at)
# VALUES(src.rating_id,src.poi_id,src.overall,src.total_rating,src.score_label,src.sources,src.breakdown,src.sentiment_avg,src.source_id,src.event_id,src.created_at,src.updated_at)
# """)

# # -----------------------------------------------------------------------------
# # ai_poi_review
# #
# # IMPORTANT:
# # Physical Gold ai_poi_review keeps the legacy/V1 column names:
# #   rating, title, review_trip_type, media_urls, helpful_votes, ...
# #
# # Silver poi_review uses normalized V2 names:
# #   rating_score, review_title, travel_type, review_media_urls, helpful_count, ...
# #
# # Map explicitly here. Do NOT use rating_score/review_title/helpful_count as
# # target column names because those columns do not exist in Gold ai_poi_review.
# # -----------------------------------------------------------------------------
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_ai_poi_review_src AS
# SELECT
#     r.review_id,
#     r.poi_id,
#     r.source_name AS source,
#     r.source_review_id,
#     r.language,
#     r.reviewer_name,
#     r.rating_score AS rating,
#     r.review_title AS title,
#     r.review_text,
#     r.review_date,
#     r.travel_type AS review_trip_type,
#     r.response_text,
#     r.responded_at,
#     to_json(r.review_media_urls) AS media_urls,
#     r.helpful_count AS helpful_votes,
#     CAST(NULL AS STRING) AS positive,
#     CAST(NULL AS STRING) AS negative,
#     e.source_id,
#     e.event_id,
#     COALESCE(
#         r.published_at,
#         CAST(r.review_date AS TIMESTAMP),
#         r._created_at
#     ) AS created_at,
#     current_timestamp() AS updated_at
# FROM poi_review r
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = r.poi_id
# LEFT JOIN poi_entity e
#     ON e.poi_id = r.poi_id
#    AND e._is_current = true
# WHERE r.is_visible = true
# """)

# spark.sql("""
# MERGE INTO sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi_review tgt
# USING v2_ai_poi_review_src src
# ON tgt.review_id = src.review_id

# WHEN MATCHED THEN UPDATE SET
#     tgt.poi_id            = src.poi_id,
#     tgt.source            = src.source,
#     tgt.source_review_id  = src.source_review_id,
#     tgt.language          = src.language,
#     tgt.reviewer_name     = src.reviewer_name,
#     tgt.rating            = src.rating,
#     tgt.title             = src.title,
#     tgt.review_text       = src.review_text,
#     tgt.review_date       = src.review_date,
#     tgt.review_trip_type  = src.review_trip_type,
#     tgt.response_text     = src.response_text,
#     tgt.responded_at      = src.responded_at,
#     tgt.media_urls        = src.media_urls,
#     tgt.helpful_votes     = src.helpful_votes,
#     tgt.positive          = src.positive,
#     tgt.negative          = src.negative,
#     tgt.source_id         = src.source_id,
#     tgt.event_id          = src.event_id,
#     tgt.updated_at        = src.updated_at

# WHEN NOT MATCHED THEN INSERT (
#     review_id,
#     poi_id,
#     source,
#     source_review_id,
#     language,
#     reviewer_name,
#     rating,
#     title,
#     review_text,
#     review_date,
#     review_trip_type,
#     response_text,
#     responded_at,
#     media_urls,
#     helpful_votes,
#     positive,
#     negative,
#     source_id,
#     event_id,
#     created_at,
#     updated_at
# )
# VALUES (
#     src.review_id,
#     src.poi_id,
#     src.source,
#     src.source_review_id,
#     src.language,
#     src.reviewer_name,
#     src.rating,
#     src.title,
#     src.review_text,
#     src.review_date,
#     src.review_trip_type,
#     src.response_text,
#     src.responded_at,
#     src.media_urls,
#     src.helpful_votes,
#     src.positive,
#     src.negative,
#     src.source_id,
#     src.event_id,
#     src.created_at,
#     src.updated_at
# )
# """)

# # -----------------------------------------------------------------------------
# # ai_enriched_poi source
# # -----------------------------------------------------------------------------
# # Source of truth for enrichment is Silver poi_enrichment.
# # One poi_id = one language, therefore the join key remains poi_id.
# #
# # IMPORTANT:
# # Do NOT project raw enrichment column names together with a.*.
# # ai_poi already contains some serving columns such as cancellation_policy.
# # Reusing the same name creates COLUMN_ALREADY_EXISTS when Spark creates the
# # temp view. Every enrichment-only projection is therefore explicitly prefixed
# # with "enrichment_".
# spark.sql("""
# CREATE OR REPLACE TEMP VIEW v2_ai_enriched_src AS
# SELECT
#     a.*,

#     e.poi_group_id                  AS enrichment_poi_group_id,
#     e.lang                          AS enrichment_lang,
#     e.enrichment_id                 AS enrichment_id,
#     e.model_name                    AS enrichment_model_name,
#     e.ref_business_sector           AS enrichment_ref_business_sector,
#     e.ref_type                      AS enrichment_ref_type,
#     e.description                   AS enrichment_description,
#     e.time_to_spend_minutes         AS enrichment_time_to_spend_minutes,
#     e.best_time_type                AS enrichment_best_time_type,
#     e.best_time_from_month          AS enrichment_best_time_from_month,
#     e.best_time_to_month            AS enrichment_best_time_to_month,
#     e.best_time_value               AS enrichment_best_time_value,
#     e.cost_vnd                      AS enrichment_cost_vnd,
#     e.cost_method                   AS enrichment_cost_method,
#     e.cost_confidence               AS enrichment_cost_confidence,
#     e.experiences_json              AS enrichment_experiences_json,
#     e.review_summary                AS enrichment_review_summary,
#     e.reasoning_short               AS enrichment_reasoning_short,
#     e.confidence                    AS enrichment_confidence,
#     e.needs_manual_review           AS enrichment_needs_manual_review,
#     e.story                         AS enrichment_story,
#     e.usp_highlights_json           AS enrichment_usp_highlights_json,
#     e.local_tips_or_neighborhood    AS enrichment_local_tips_or_neighborhood,
#     e.cancellation_policy           AS enrichment_cancellation_policy,
#     e.enriched_at                   AS enrichment_enriched_at

# FROM sgr_visitvn_dev.lh_vv_gold.dbo.ai_poi a
# INNER JOIN v2_impacted_poi i
#     ON i.poi_id = a.poi_id
# LEFT JOIN poi_enrichment e
#     ON e.poi_id = a.poi_id
#    AND e.is_current = true
# """)

# # Fail fast if a future schema change introduces duplicate columns again.
# _ai_enriched_cols = spark.table("v2_ai_enriched_src").columns
# _ai_enriched_dups = sorted({
#     c for c in _ai_enriched_cols
#     if _ai_enriched_cols.count(c) > 1
# })

# if _ai_enriched_dups:
#     raise RuntimeError(
#         "NB50_AI_ENRICHED_DUPLICATE_COLUMNS: " + str(_ai_enriched_dups)
#     )

# print(
#     "NB50 v2_ai_enriched_src = OK; "
#     f"columns={len(_ai_enriched_cols)}"
# )

# # Keep current physical ai_enriched_poi columns; MERGE by poi_id.
# # If/when ai_enriched_poi is materialized in V2, map these explicitly-prefixed
# # enrichment fields to its physical target schema instead of using SELECT *.

# notebookutils.notebook.exit("SUCCESS")
